### Q1. YouTube 트렌딩 영상 데이터셋에서, 날짜순으로 정렬했을 때, 전날에도 트렌딩에 있었고 dislike 수가 전일보다 증가한 날이 가장 길게 이어진 영상의 channelTitle을 출력하라 <br>
난이도(상)

In [65]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedevastator/youtube-trending-videos-dataset")
df = pd.read_csv(path + "/youtube.csv")

print(df)

         index     video_id trending_date  \
0            0  2kyS6SvSYSE      17.14.11   
1            1  1ZAPwfrtAFY      17.14.11   
2            2  5qpjK5DgCt4      17.14.11   
3            3  puqaWrEC7tY      17.14.11   
4            4  d380meD0W0M      17.14.11   
...        ...          ...           ...   
161465  161465  sGolxsMSGfQ      18.14.06   
161466  161466  8HNuRNi8t70      18.14.06   
161467  161467  GWlKEM3m2EE      18.14.06   
161468  161468  lbMKLzQ4cNQ      18.14.06   
161469  161469  POTgw38-m58      18.14.06   

                                                    title  \
0                      WE WANT TO TALK ABOUT OUR MARRIAGE   
1       The Trump Presidency: Last Week Tonight with J...   
2       Racist Superman | Rudy Mancuso, King Bach & Le...   
3                        Nickelback Lyrics: Real or Fake?   
4                                I Dare You: GOING BALD!?   
...                                                   ...   
161465                       HOW

shift 만 사용 (X)

In [ ]:
df_sort = df.sort_values(by=['title','trending_date'],ascending=True)
df_sort['trending_date'] = pd.to_datetime(df_sort['trending_date'],format='%y.%d.%m')

df_sort['pre_trending'] = df_sort['trending_date'].shift(1)
df_sort['pre_dislikes'] = df_sort['dislikes'].shift(1)
df_sort['pre_title'] = df_sort['title'].shift(1)
trend_mask = (df_sort['trending_date'] - df_sort['pre_trending'])==pd.Timedelta(1,"D")
dislikes_mask = (df_sort['dislikes']>df_sort['pre_dislikes'])
title_mask = df_sort['pre_title'] == df_sort['title']

answer_df = df_sort[trend_mask & dislikes_mask & title_mask]

groupby 사용
mask를 따로 만들기보다 새로운 column 추가하는 형태의 작업 필수

True를 카운트 하는 방법 </br>

장점: 연속된 길이를 쉽게 카운트 할 수 있다. </br>
단점: 연속이 끊어질 때 카운트를 끊어야 한다 </br>
</br>
</br>
False를 카운트 하는 방법 </br>
장점: 같은 그룹으로 묶기 좋다. </br>
단점: 가장 큰 그룹을 찾고 그 그룹의 channeltitle을 출력하는 노고가 필요하다.



In [ ]:
df['trending_date'] = pd.to_datetime(df['trending_date'], format='%y.%d.%m')
df_sort = df.sort_values(by=['title','trending_date'], ascending=True)

df_sort['trending_diff'] = df_sort.groupby('title')['trending_date'].diff()

# groupby 사용 용례
# print(df_sort['dislikes_False'].groupby(df_sort['channel_title']).size())
# print(df_sort.groupby(df_sort['channel_title'])['dislikes_False'].size())
# groupby에 as_index개념 활용하면 골치가 반으로 줄어듬

#어제와 연속하면서 dislikes 증가
df_sort['dislikes_up'] = (df_sort['trending_diff'] == pd.Timedelta(days=1)) & (df_sort.groupby('title')['dislikes'].diff() > 0)

# dislike가 증가하지 않았으면 숫자 증가. 증가중인 그룹에 같은 숫자 부여 (그룹화)
df_sort['dislikes_False'] = (df_sort['dislikes_up'] == False).cumsum()

# 두 값이 동등하게 나왔으나 알고보니 공동 1등 둘 있음
df_sort.groupby(['dislikes_False','title','channel_title']).size().sort_values(ascending=False).iloc[:5]
df_sort.groupby(['dislikes_False','channel_title'],as_index=False).size().sort_values(by='size',ascending=False).iloc[:5]

# 공동 1등 동시 추출
answerlist = df_sort.groupby(['dislikes_False', 'channel_title'], as_index=False).size().sort_values(by='size',ascending=False)
answer_max = answerlist.loc[answerlist['size'] >= answerlist.max().loc['size']]
print(answer_max['channel_title'].to_list())



# # 쓸모없는거 - as_index라는 좋은 물건을 몰랐을 떄나 하던 짓
# df_True = df_sort[df_sort['dislikes_up'] == True]
# df_True.groupby(['dislikes_False'])['channel_title']

# as_index 안쓸 시
# #index 사용법
# answer1 = df_True.groupby(['title','channel_title']).size().sort_values(ascending=False).index[0][1]

# #idxmax, idxmin 사용법
# answer2 = df_True.groupby(['title','channel_title']).size().sort_values(ascending=False).idxmax()[1]

# # index -> data로 변환
# answer3 = df_True.groupby(['title','channel_title']).size().sort_values(ascending=False).reset_index().iloc[0, 1]



#answer4 = df_True.groupby('')

#한줄 정리
# result = df_sort[df_sort['dislikes_up']].groupby(['channel_title', 'dislikes_False']).size().idxmax()[0]
# print(result)





['FiftyShadesVEVO', 'TWICE JAPAN OFFICIAL YouTube Channel']
       dislikes_False                         channel_title  size
30026           72630                       FiftyShadesVEVO    36
25851           63087  TWICE JAPAN OFFICIAL YouTube Channel    36
14849           35102                            KarolGVEVO    35
9416            23390                              Flo Rida    35
862              2603                               2CELLOS    35
...               ...                                   ...   ...
32670           82700                           AstronoGeek     1
32669           82699                           AstronoGeek     1
32668           82696                           AstronoGeek     1
32667           82695                           AstronoGeek     1
32683           82738                               Deja Vu     1

[32714 rows x 3 columns]


#### 유사 문제 결과 풀이

In [ ]:
import pandas as pd

df = pd.read_csv('/data/48bd152a.csv')

df['trending_date2'] = pd.to_datetime(df['trending_date2'], format='%Y-%m-%d')
df_sort = df[['channelTitle', 'title','trending_date2','dislikes']].sort_values(by=['title','trending_date2'], ascending=True)
df_sort['dislikes_up'] = (df_sort.groupby(['title'])['trending_date2'].diff() == pd.Timedelta(1,'D'))&(df_sort.groupby(['channelTitle'])['trending_date2'].diff() == pd.Timedelta(1,'D')) & (df_sort.groupby(['title'])['dislikes'].diff() > 0 )


# 연속된 차수끼리 그룹화
df_sort['group'] = (df_sort['dislikes_up']== False).cumsum()
df_group = df_sort.groupby(['title','channelTitle','group'],as_index=False).size().sort_values('size', ascending=False)


# 문자열로 반환하려면 : 같은거 쓰지말고 좌표 정확하게 잡아서 출력할 것
answer = df_group.iloc[0,1]

print(answer)

### Q2. 논란으로 인기동영상이 된 케이스를 확인하고 싶다. dislikes수가 like 수보다 높은 동영상을 제작한 채널들 중 두 번째 채널명을 출력하라 <br>
난이도<하>

In [59]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedevastator/youtube-trending-videos-dataset")
df = pd.read_csv(path + "/youtube.csv")

groupby와 max로 중복 채널을 없애고 dislike 순으로 채널을 출력하였음 <br>
다만 답안으로는 list화 하여 unique 사용을 권장하는 것으로 보임

In [64]:
#모범답안
answer = list(df.loc[df['likes'] < df['dislikes']]['channel_title'].unique())  
#loc을 사용하면 대용량데이터를 더 빠르게 탐색 가능. 근데 시험때는 굳이 넣을 필요 없을 듯?

# df 빼먹지 말 것
answer2 = df[df['likes'] < df['dislikes']]['channel_title'].unique()


print(answer2[1])


NBA Highlights Â· YouTube


시행착오 리스트

In [ ]:
# 알 수 없는 경고가 계속 뜸 and 너저분함
df_filter = df[['likes','dislikes','channel_title']]
df_filter['diff'] = df_filter['dislikes'] - df_filter['likes']
df_true = df_filter[df_filter['diff'] > 0]
df_true = df_true['channel_title'].unique()
print(df_true[1])

#groupby로만작성한 결과 (두번째 채널 출력하라는 부분을 오인함)
df_group = df_true.groupby(['channel_title'], as_index=False)['dislikes'].max().sort_values('dislikes',ascending=False)
top_2=df_group.nlargest(2,'dislikes',keep='all')

print(top_2.iloc[1,0])

### Q3. 채널명을 바꾼 케이스가 있는지 확인하고 싶다. channelId의 경우 고유값이므로 이를 통해 채널명을 한번이라도 바꾼 채널의 개수를 구하여라 <br>
난이도(하)

In [75]:
import pandas as pd
import kagglehub

# Download latest version
# 실습용 가상 데이터 생성
df = {
    'channelId': [
        'UC_001', 'UC_001', 'UC_001',  # 채널 1
        'UC_002', 'UC_002',            # 채널 2
        'UC_003', 'UC_003', 'UC_003'   # 채널 3
    ],
    'channelTitle': [
        'Tech Review', 'Tech Review', 'Tech Review Pro', # 이름을 1번 변경함
        'Daily Vlog', 'Daily Vlog',                      # 이름을 변경하지 않음
        'Gaming Zone', 'Game Zone', 'GZ Official'        # 이름을 2번 변경함
    ],
    'views': [1000, 1500, 3000, 500, 600, 2000, 2500, 4000]
}
df = pd.DataFrame(df)


nunique() 와 count()의 차이

In [83]:
df_filter = df[['channelId', 'channelTitle']]
df_filter1 = df_filter.groupby('channelId',as_index=False).nunique().sort_values('channelTitle', ascending=False)
answer1 = df_filter1[df_filter1['channelTitle']>1]['channelId'].count()
print(answer1)



# 중복된 케이스를 짚어내기 힘듬
df_filter2 = df_filter.groupby('channelId',as_index=False)['channelId'].value_counts()
answer2 = df_filter2[df_filter2['count']>1]['channelId'].count()
print(answer2)



2
3


### Q4. trending_date의 일요일 데이터만 사용하여 categoryId별 좋아요 수 중앙값, 평균 조회수, 영상 수를 구하라. 좋아요 수 중앙값이 전체 categoryId 중앙값 이상이고 영상 수가 평균 이상인 categoryId만 남긴 뒤, 평균 조회수 내림차순으로 정렬했을 때 첫 번째 categoryId 값을 출력하라 <br>
난이도(상)

In [1]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedevastator/youtube-trending-videos-dataset")
df = pd.read_csv(path + "/youtube.csv")

c:\Users\daeye\anaconda3\envs\bigdata\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


모범답안

In [ ]:
df['trending_date'] = pd.to_datetime(df['trending_date'], format='%y.%d.%m')

df['dayofweek'] = df['trending_date'].dt.dayofweek  # 요일을 숫자로 출력
df['day_name'] = df['trending_date'].dt.day_name()   # 요일을 영어로 출력
df_filter = df[df['dayofweek'] == 6]
stats = df_filter.groupby('category_id').agg(median_likes=('likes', 'median'), mean_views=('views', 'mean'), video_count=('title', 'size'))

시행착오

In [ ]:
df['trending_date'] = pd.to_datetime(df['trending_date'], format='%y.%d.%m')

df['dayofweek'] = df['trending_date'].dt.dayofweek  # 요일을 숫자로 출력
df['day_name'] = df['trending_date'].dt.day_name()   # 요일을 영어로 출력
df_filter = df[df['dayofweek'] == 6]

df_filter = df_filter[['category_id','title','likes','views']]


category_median = df_filter.groupby('category_id')['likes'].median()
category_likes = df_filter.groupby('category_id')['views'].mean()
category_title = df_filter.groupby('category_id')['title'].nunique()

# print(category_median)
# print(category_likes)
# print(category_title)

# [[category별 좋아요 중앙값 > 전체 category_id 좋아요 중앙값 이상] & [category_i별 영상수 > 영상수 평균]] sort_values(ascending=False)인 category_id.iloc[0] print
# '전체 category_id 좋아요 중앙값'의 의미가 뭔지 고민 좀 함


# 필터대상.map(조건)
df_filter['filter'] = df_filter['category_id'].map((category_median > category_median.median()) & (category_title >= category_title.mean()))

answer = df_filter[df_filter['filter']==True].sort_values('views', ascending=False)


# loc 사용법. 인덱스 문제, reset_index(drop=True)를 사용해서 인덱스를 재정렬
# answer.reset_index(drop=True).loc[0,'category_id']

# iloc
answer.iloc[0,0]




# 심심해서 써넣은 columns drop 코드.
#df_filter = df_filter.drop(columns=['index','video_id','video_error_or_removed'])


### Q5. 각 요일별 인기 영상들의 category_id는 각각 몇개 씩인지 하나의 데이터프레임을 정리했을 때 두 번째 행의 개수를 출력하라<br>
난이도:(중)

In [42]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedevastator/youtube-trending-videos-dataset")
df = pd.read_csv(path + "/youtube.csv")

count와 list를 사용한 내 풀이 방식. 데이터프레임화가 생략됨

In [99]:
df['trending_date'] = pd.to_datetime(df['trending_date'], format='%y.%d.%m')

df['day_name'] = df['trending_date'].dt.day_name()

# DataFrame화 필요
# df.groupby(['day_name','category_id'])['category_id'].count()

# as_index를 통한 DataFrame화
# df.groupby(['day_name','category_id'] as_index=False)['category_id'].count()

# reset_index를 통한 DataFrame화. 멀티 인덱싱 x. 결과물을 category_id가 아닌 count열에 쓰도록 명시. 안그러면 중복으로 에러
df.groupby(['day_name','category_id'])['category_id'].count().reset_index(name='count')

# 대괄호 두개가 멀티 인덱싱유지하고 DataFrame화. type 찍어보면 나옴
# df.groupby(['day_name', 'category_id'])[['category_id']].count()

# to_frame
# print(df.groupby(['day_name','category_id'])['category_id'].size().to_frame())


answer = df.groupby(['day_name','category_id'], as_index=False)['category_id'].count()


print(answer)
print(df.groupby(['day_name','category_id'])['category_id'].count().reset_index(name='count'))
# print(df.groupby(['day_name', 'category_id'])[['category_id']].count())

      day_name  category_id
0       Friday         1303
1       Friday          279
2       Friday         3903
3       Friday          296
4       Friday         1520
..         ...          ...
116  Wednesday          542
117  Wednesday          695
118  Wednesday           48
119  Wednesday            2
120  Wednesday           47

[121 rows x 2 columns]
      day_name  category_id  count
0       Friday            1   1303
1       Friday            2    279
2       Friday           10   3903
3       Friday           15    296
4       Friday           17   1520
..         ...          ...    ...
116  Wednesday           27    542
117  Wednesday           28    695
118  Wednesday           29     48
119  Wednesday           30      2
120  Wednesday           43     47

[121 rows x 3 columns]


모범답안. size로 원하는 값을 추출하고 pivot으로 데이터프레임화 시키는 과정

In [ ]:
df = pd.read_csv('/data/d085cf2b.csv')

df['trending_date2'] = pd.to_datetime(df['trending_date2'])

group = df.groupby([df['trending_date2'].dt.day_name(), 'categoryId'], as_index=False).size()

answer = group.pivot(index='categoryId', columns='trending_date2')

answer = answer['size', 'Friday'].iloc[1]

print(answer)

### Q6 view_count가 0이 아닌 영상만 남긴 뒤 comment_count를 view_count로 나눈 비율을 구하라. 이 비율이 가장 높은 영상의 title을 출력하라<br>
난이도:(하)

In [114]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedevastator/youtube-trending-videos-dataset")
df = pd.read_csv(path + "/youtube.csv")

In [ ]:
df_filter = df[df['views'] != 0]
df_filter = df_filter[['title', 'comment_count','views']]

df_filter['per'] = df_filter['comment_count'] / df_filter['views']
answer = df_filter.sort_values('per', ascending=False).iloc[0,0]
print(answer)



# 잘못된 예
# print(list(df_filter[['views']==0]))
# 의도 = 0인게 남아있는지 확인
# df_filter에 sum, any, min 등의 함수를 붙여서 해결해야했고 []와 ()를 잘못 혼용함

# 했던 실수
# 위에서 df_filter로 선언해놓고 밑에서 그냥 df를 써놓은 탓에 view!=0이 동작을 안했음

...â¤


### Q7 view_count가 0이 아니고 댓글수 대비 조회수 비율이 0이 아닌 영상만 남겨라. comment_count를 view_count로 나눈 비율이 가장 낮은 영상의 title을 출력하라<br>
난이도:(하)

In [136]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedevastator/youtube-trending-videos-dataset")
df = pd.read_csv(path + "/youtube.csv")

In [142]:
df_filter = df[['title','views','comment_count']]
df_filter = df_filter[df_filter['views']!=0]
df_filter['per'] = df_filter['comment_count'] / df_filter['views']
df_filter = df_filter[df_filter['per']!=0].sort_values('per', ascending=True)
print(df_filter.iloc[0,0])


Coachella 2018 LIVE Channel 1


### Q8 like 대비 dislike의 수가 가장 적은 영상의 title은 무엇인가? (like, dislike 값이 0인경우는 제외한다) <br>
난이도:(하)

In [ ]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedevastator/youtube-trending-videos-dataset")
df = pd.read_csv(path + "/youtube.csv")

In [13]:
df_filter = df[['title', 'likes', 'dislikes']]
df_filter = df_filter[(df_filter['likes'] !=0) & (df_filter['dislikes'] != 0)]

# check
# print(df_filter[df_filter['dislikes']==0]['dislikes'].count())

df_filter['ratio'] = df_filter['dislikes'] / df_filter['likes']

answer = df_filter.sort_values('ratio', ascending=True).iloc[0,0]

print(answer)



Swing - Rivage (Prod. Le Motel)


### Q9. 가장많은 트렌드 영상을 제작한 채널명은 무엇인가? (날짜기준, 중복포함)<br>
난이도:(하)

In [ ]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedevastator/youtube-trending-videos-dataset")
df = pd.read_csv(path + "/youtube.csv")

첫번째 방식

In [38]:
df_filter = df[['channel_title', 'title','trending_date']]

df_filter['trending_date'] = pd.to_datetime(df_filter['trending_date'], format='%y.%d.%m')

df_filter.sort_values('trending_date')


answer1 = df_filter.groupby(['channel_title'], as_index=False)['title'].nunique().reset_index(drop=True)
print(answer1.sort_values('title',ascending=False).iloc[0,0])

The Late Show with Stephen Colbert


두번째 방식 (agg 사용)

In [56]:
df_filter = df[['channel_title', 'title','trending_date']]

df_filter['trending_date'] = pd.to_datetime(df_filter['trending_date'], format='%y.%d.%m')

answer = df_filter.groupby(['channel_title'],as_index=False).agg(num=('title','nunique')).sort_values('num',ascending=False)

print(answer)


                            channel_title  num
9546   The Late Show with Stephen Colbert  209
10324                           VikatanTV  208
2827                     Elhiwar Ettounsi  201
2752                                 ESPN  184
1540                                  CNN  170
...                                   ...  ...
12347                     ì—°í•©ë‰´ìŠ¤ TV    1
12346       ì—¬ìžì¹œêµ¬ GFRIEND OFFICIAL    1
12343                              ìš°ì°Œ    1
12342         ìŠ¤ë¸ŒìŠ¤ë‰´ìŠ¤ SUBUSU NEWS    1
12341                  ì°½ì¡°ì˜ê°í´ëŸ½    1

[12361 rows x 2 columns]


답안. 엉뚱하게도 핵심 조건은 채널ID의 value count를 사용하여 검증한 채널 명이었음

In [ ]:
df = pd.read_csv('/data/093deca9.csv')

answer = df.loc[df['channelId'] == df['channelId'].value_counts().index[0]]['channelTitle'].unique()[0]

print(answer)

### Q10 20회(20일) 이상 인기동영상 리스트에 포함된 영상 수를 출력하라<br>
난이도:(중) <br>
부록: size와 size()의 차이

size와 size()의 차이

size는 완성된 자료인 Dataframe의 속성 참조, 접근
size()는 대기 상태인 groupby에서의 매서드 호출

In [84]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedevastator/youtube-trending-videos-dataset")
df = pd.read_csv(path + "/youtube.csv")

정답 근사치.
다만 title이 같은데 다른 영상인 경우를 대비하여 채널 ID등을 체크하지 않아 틀렸음

In [83]:
df_filter = df.groupby(['title']).agg(size=('title','size'))
df_filter = df_filter[df_filter['size']>=20]

print(df_filter.size)


1049


정정 코드<br>
원 예제에서는 video_id가 존재하지 않고 channel_id만 존재

In [98]:
df_filter = df.groupby(['channel_title','title']).agg(size=('title', 'size')).sort_values('size', ascending=False)
df_filter = df_filter[df_filter['size']>=20]

answer = df_filter.size
print(df_filter.size)


1040


예시 답안

In [100]:
answer = (df[['title', 'channel_title']].value_counts() >= 20).sum()

print(answer)

1040


### Q11 ct 컬럼을 datetime으로 변환한 뒤 videoname별 데이터 개수를 구하라. 데이터 개수가 두 번째로 많은 videoname의 개수를 출력하라<br>
난이도:(하)

In [3]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kukuroo3/youtube-episodic-contents-kr")
channel = pd.read_csv(path + "/channelInfo.csv")
video = pd.read_csv(path + '/videoInfo.csv')

문제 잘볼것. videoname이 아니라 '개수' 출력임.

In [18]:
video['ct'] = pd.to_datetime(video['ct'], format=('%Y-%m-%d %H:%M:%S'))
video_sort = video.groupby(['videoname'], as_index=False).agg(size=('videoname', 'value_counts')).sort_values('size', ascending=False)

print(video_sort)
print(video_sort.iloc[1,1])

  videoname  size
0    공범 EP1  3492
1    공범 EP2  3204
2    공범 EP3  2568
3    공범 EP4  2280
4    공범 EP5  1562
5    공범 EP6  1274
6    공범 EP7   555
7    공범 EP8   266
3204


### Q12 수집된 각 video의 가장 최신화 된 날짜의 viewcount값을 정리된 결과에서 세 번째 행의 viewcount값을 출력하라<br>


In [61]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kukuroo3/youtube-episodic-contents-kr")
channel = pd.read_csv(path + "/channelInfo.csv")
video = pd.read_csv(path + '/videoInfo.csv')

video

,videopk,viewcnt,likecnt,dislikecnt,favoritecnt,cmcnt,ct,videoname
0,c5JQp6xafqc,1667010,30474,706,0,6587,2021-10-10 15:20:03,공범 EP1
1,c5JQp6xafqc,1669089,30495,707,0,6589,2021-10-10 15:30:03,공범 EP1
2,c5JQp6xafqc,1674759,30522,711,0,6596,2021-10-10 15:40:02,공범 EP1
3,c5JQp6xafqc,1677026,30555,712,0,6604,2021-10-10 15:50:03,공범 EP1
4,c5JQp6xafqc,1681824,30585,713,0,6600,2021-10-10 16:00:03,공범 EP1
...,...,...,...,...,...,...,...,...
15196,yZt-h-KcmUE,1285474,25582,3615,0,31603,2021-11-01 14:50:06,공범 EP8
15197,yZt-h-KcmUE,1286376,25592,3617,0,31627,2021-11-01 15:00:05,공범 EP8
15198,yZt-h-KcmUE,1287172,25597,3619,0,31643,2021-11-01 15:10:05,공범 EP8
15199,yZt-h-KcmUE,1288138,25606,3619,0,31648,2021-11-01 15:20:04,공범 EP8


unique()로 작업. sort를 건드려선 안됨

In [105]:
video_filter = video.sort_values('ct', ascending=False)
video_filter = video_filter.groupby('videoname',as_index=False).agg(first=('viewcnt','first'))

print(video_filter.iloc[2,1])

1671294


그룹별 n등 뽑아내기 새로운걸 배울 수 있었지만 sort를 건드려선 안됨

In [91]:
video_filter = video.sort_values('ct', ascending=False)

video_filter['rank']=video_filter.groupby('videoname').cumcount()

video_filter = video_filter[video_filter['rank']==0]
video_filter = video_filter.sort_values('videoname', ascending=True)
answer = video_filter.iloc[2,1]

print(answer)

1671294


AI가 추천한 idxmax() 기법 이것도 새로운걸 배울 수 있었지만 sort를 건드려선 안됨

In [92]:
video_filter = video.loc[video.groupby('videoname')['ct'].idxmax(), ['videoname', 'viewcnt']].reset_index(drop=True)

print(video_filter.iloc[2])

videoname     공범 EP3
viewcnt      1671294
Name: 2, dtype: object


duplicate 사용. drop_duplicate도 사용해볼것.